In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://finance.yahoo.com/markets/crypto/most-active/"
# there are different request methods: get (most popular and easy)
# stateful protocol - connection established and then closed
# stateless protocol - http, every request (like each other page opening) is separate.
# all data is sent in headers along with URL. Like session id to identify next request and source of accessing web page
# HTTP response status codes
# 100s -
# 200s - OK
# 300s - redirect
# 400s - client error (404 - not found, 429 - too many requests)

HEADERS = {'User-Agent': 'Mozilla/5.0'} #enough for the site to wok

def parse_human_readable(value):
    if not value:
        return None
    val = value.replace(',', '')
    last_char = val[-1]
    val_no_last_char = val[:-1]

    match last_char:
        case 'K':
            return float(val_no_last_char)*1000
        case 'M':
            return float(val_no_last_char)*1_000_000
        case 'B':
            return float(val_no_last_char)*1_000_000_000
        case 'T':
            return float(val_no_last_char)*1_000_000_000_000
        case _:
            return float(val)

def crypto_volumes():
    response = requests.get(URL, headers=HEADERS) #question to get the info from URL
    print(response)
    print(response.status_code)
    html = response.text
    #print(html)
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find('table', attrs={'class': 'yf-19xszjc'}) #(this code changes once every while, so in some time it might no longer work)
    print(table)
    rows = table.find('tbody').find_all('tr')
    data = [] #to turn it into pandas dataframe
    for row in rows:
        print(row)
        print('-------------')
        cells = row.find_all('td')
        symbol = cells[0].text.strip()
        print(symbol)
        price = cells[4].text.strip()
        market_cap = cells[6].text.strip()
        volume = cells[8].text.strip()
        record = {
            'symbol': symbol,
            'price': parse_human_readable(price),
            'market_cap': market_cap,
            'volume': volume,
        }
        print(record)
        data.append(record)
    dff = pd.DataFrame(data)
    dff['market_cap'] = dff['market_cap'].apply(parse_human_readable)
    dff['volume'] = dff['volume'].apply(parse_human_readable)
    print(dff)
    return dff

def save_crypto_volumes(dff):
    dff.to_csv('data/crypto_volumes.csv', index=False)
    dff.to_json('data/crypto_volumes.json', orient='records', indent=4)
    dff.to_excel('data/crypto_volumes.xlsx', index=False)

In [2]:
def x2(x):
    return x*2

print(x2(1000))
x_2 = lambda x:x*2 #lambda expression shows there is a function that turns x into x*2
print(x_2(1000))

2000
2000


In [3]:
import plotly.express as px

def human_readable(value):
    return f'{value/1_000_000_000}B'

def plot_volume(dff):
    dff['market_capitalization'] = dff['market_cap'].apply(human_readable)
    dff_big = dff[dff['market_cap'] > 1_000_000_000]
    fig = px.bar(dff_big, x='symbol', y='volume', title='Crypto Volume', color='market_cap',
                 hover_data=['price','market_cap','volume'])
    fig.show()

In [5]:
df = crypto_volumes()
save_crypto_volumes(df)
plot_volume(df)
#protocol = set of rules to communicate
#this is manually hard to read, extracting data via BeautifulSoup

<Response [200]>
200
<table class="yf-19xszjc bd"><thead class="yf-19xszjc"><tr class="yf-19xszjc"> <th class="[&amp;_.symbol]:tw-text-md yf-19xszjc so lpin shad" data-testid-header="ticker"><div class="colCont yf-19xszjc"> Symbol </div> </th> <th class="leftAlignHeader companyName yf-19xszjc" data-testid-header="companyshortname.raw"><div class="colCont yf-19xszjc"> Name </div> </th> <th class="yf-19xszjc" data-testid-header="sparkline"><div class="colCont yf-19xszjc"> <span data-svelte-h="svelte-1uypmr0"> </span> </div> </th> <th class="yf-19xszjc so" data-testid-header="full_day_price"><div class="colCont yf-19xszjc"> <div class="container yf-1d5e06g"><span>Price</span> </div> </div> </th> <th class="yf-19xszjc so" data-testid-header="full_day_change"><div class="colCont yf-19xszjc"> Change </div> </th> <th class="yf-19xszjc so" data-testid-header="full_day_change_percent"><div class="colCont yf-19xszjc"> Change % </div> </th> <th class="yf-19xszjc so" data-testid-header="intradayma